In [ ]:
from pathlib import Path
import pandas as pd
from io import StringIO
import re
from unidecode import unidecode
import numpy as np

def spliter_voisin(text):
    separators = [';', ',', ':', '/','et']

    regex_pattern = '|'.join(map(re.escape, separators))

    return re.split(regex_pattern, text)[0]

def match_number(text):
    return bool(re.fullmatch(r'\d{10}', text.replace(' ', '')))

def normalize(name):
    return '_'.join(str(name).lower().split()).strip()

def denormalize(name):
    return ' '.join(str(name).capitalize().split('_')).strip()

file_output = r"C:\Users\L14\Desktop\controle_voisins_05_12.xlsx"
#file_output = r"C:\Users\L14\Desktop\controle_group_2_19_11.xlsx"
folder_path = Path(r"C:\Users\L14\Desktop\PRESFOR\VANIE BI")
voisins_folder_path = Path(r"C:\Users\L14\Downloads\id_voisin_unique")

voisins_files_path = voisins_folder_path.rglob('*.csv')
files_path = folder_path.rglob('*.csv')

bd_voisins = pd.DataFrame()

for file in voisins_files_path:
    df = pd.read_csv(file)
    bd_voisins = pd.concat([bd_voisins,df])

print(bd_voisins.shape)
for file in files_path:
    with open(file, "r", encoding='utf-8') as f:
        lines = f.readlines()

        tables = {}
        current_table_name = None
        current_table_lines = []
        tables_titles = []
        for line in lines:
            line = line.strip()

            if line == "":
                continue

            if line.lower().startswith("table:"):
                
                if current_table_name and current_table_lines:
                    csv_block = "\n".join(current_table_lines)
                    df = pd.read_csv(StringIO(csv_block))
                    tables[current_table_name] = df
                    current_table_lines = []
                
                current_table_name = line.split(":", 1)[1].strip() 
                tables_titles.append(current_table_name)
            else:
                current_table_lines.append(line)

            
        if current_table_name and current_table_lines:
            csv_block = "\n".join(current_table_lines)
            df = pd.read_csv(StringIO(csv_block))
            tables[current_table_name] = df

        print(tables_titles)
        
        df_land = tables['land'][['code','label','description','village','numParcelleOF']]
        df_observation_presence = tables['observationPresence'][['code','memberType','nameOfPerson','numberCNI','signatoryPhoto','presenceDate','representant']]
        df_observation_neighborhood = tables['observationNeighborhood'][['code','position','nameOfNeighbor','isPresent','presenceDate','description']]
        df_observation = tables['observation'][['codeObservation','creationDate','observationDate','label','description','presences','neighborhood']]
        df_applicant = tables['applicant'][['applicantNumber','firstname','lastname','nationality','phoneNumber','nameOfGroup','typeOfIndividualCertificate','numParcelleOF']]

        demandes_tablette = []
        demandes_tablette = df_land['code'].values

        df_observation.loc[:,'codeObservation'] = df_observation['codeObservation'].str.replace('PVCL','')
        df_observation.set_index(keys='presences',drop='False',inplace=True)

        df_land.loc[:,'code'] = df_land['code'].str.replace('LAN','')
        df_land = df_land.drop_duplicates(subset='code',keep='first')
        df_land.set_index(keys='code',drop='False',inplace=True)

        df_applicant.loc[:,'applicantNumber'] = df_applicant['applicantNumber'].str.replace('APT','')
        df_applicant = df_applicant.drop_duplicates(subset='applicantNumber',keep='first')
        df_applicant.set_index(keys='applicantNumber',drop='False',inplace=True)

        df_observation_presence = df_observation_presence.copy()
        df_observation_presence['code'] = df_observation_presence['code'].apply(lambda row: df_observation.loc[row]['codeObservation'])
        df_observation_presence['numParcelleOF'] = df_observation_presence['code'].apply(lambda row: df_land.loc[row]['numParcelleOF'])
        df_observation_presence['nameOfApplicant'] = df_observation_presence['code'].apply(lambda row: df_applicant.loc[row]['firstname'] + ' ' + df_applicant.loc[row]['lastname'])
        df_observation_presence['numberPhoneOfApplicant'] = df_observation_presence['code'].apply(lambda row: df_applicant.loc[row]['phoneNumber'])

        df_filtered = df_observation_presence[(df_observation_presence['memberType'] == 7) | ((df_observation_presence['memberType'] == 13) & df_observation_presence['representant'].astype(str).str.lower().str.contains('voisin'))].copy()
        df_filtered_ctb = bd_voisins[bd_voisins['num_demande'].isin(demandes_tablette)]
        df_filtered_ctb.loc[:,'nom_prenoms_vrais'] = ''
        df_filtered_ctb.to_excel(f'{str(file).split('.')[0]}_ctb.xlsx')
        df_filtered[['code','numParcelleOF','nameOfApplicant','numberPhoneOfApplicant','nameOfPerson','numberCNI','representant']].to_excel(f'{str(file).split('.')[0]}.xlsx')
